# Animagine XL 4.0 - Kaggle Dual GPU (T4 x2)

1. ตั้งค่า Notebook ให้เปิด Internet
2. เลือก Accelerator เป็น GPU T4 x2
3. รันเซลล์ที่ 1 เพื่ออัปเดตแพ็กเกจ
4. รันเซลล์ที่ 2 เพื่อโหลดโมเดลลง GPU ทั้ง 2 ตัวและเปิดหน้า UI


In [ ]:
# Cell 1: อัปเดตเฉพาะแพ็กเกจที่จำเป็น (ใช้ร่วมกับ Python 3.13 ของ Kaggle ได้เลย)
%pip install -q -U diffusers transformers accelerate safetensors gradio
print('ติดตั้งเสร็จแล้ว พร้อมใช้งาน!')

In [ ]:
# Cell 2: โหลดโมเดลลง GPU 2 ตัวและเปิดหน้า UI (สูงสุด 4 ภาพ)
import os
import random
import torch
import gradio as gr
from concurrent.futures import ThreadPoolExecutor
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '10'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'

if not torch.cuda.is_available():
    raise RuntimeError('ไม่พบ GPU - ไปที่ Notebook Settings แล้วเลือก Accelerator เป็น GPU T4 x2')

gpu_count = torch.cuda.device_count()
print(f'พบ GPU {gpu_count} ตัว: ' + ', '.join(torch.cuda.get_device_name(i) for i in range(gpu_count)))

model_id = 'cagliostrolab/animagine-xl-4.0'

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

load_kwargs = {
    'torch_dtype': torch.float16,
    'use_safetensors': True,
    'custom_pipeline': 'lpw_stable_diffusion_xl',
    'add_watermarker': False,
}
if hf_token:
    load_kwargs['token'] = hf_token

pipes = []
try:
    for i in range(gpu_count):
        print(f'กำลังโหลดโมเดลลง GPU {i} ({torch.cuda.get_device_name(i)})...')
        p = StableDiffusionXLPipeline.from_pretrained(model_id, **load_kwargs)
        p.scheduler = EulerAncestralDiscreteScheduler.from_config(p.scheduler.config)
        p.vae.enable_slicing()
        p.vae.enable_tiling()
        p.to(f'cuda:{i}')
        pipes.append(p)
except Exception as exc:
    raise RuntimeError(
        'โหลด Hugging Face model ไม่สำเร็จ - ตรวจว่า Kaggle เปิด Internet, HF_TOKEN ถูกต้อง, '
        'และยอมรับเงื่อนไขที่ https://huggingface.co/cagliostrolab/animagine-xl-4.0 แล้ว'
    ) from exc

print(f'โหลดโมเดลลง GPU ทั้ง {len(pipes)} ตัวสำเร็จ! พร้อมสร้างภาพ')

def _render_single(gpu_idx, prompt, negative_prompt, width, height, steps, cfg_scale, img_seed):
    device_str = f'cuda:{gpu_idx}'
    generator = torch.Generator(device=device_str).manual_seed(img_seed)
    with torch.inference_mode():
        res = pipes[gpu_idx](
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=width,
            height=height,
            num_inference_steps=int(steps),
            guidance_scale=float(cfg_scale),
            generator=generator,
        )
    with torch.cuda.device(gpu_idx):
        torch.cuda.empty_cache()
    return res.images[0], img_seed

def generate_anime_v4(prompt, negative_prompt, aspect_ratio, num_images, steps, cfg_scale, seed):
    sizes = {
        'จัตุรัส (1:1)': (1024, 1024),
        'แนวตั้งมาตรฐาน (3:4)': (896, 1152),
        'แนวตั้ง 9:16': (832, 1216),
        'แนวนอน 16:9': (1216, 832),
    }
    width, height = sizes.get(aspect_ratio, (896, 1152))
    num_images = int(num_images)
    base_seed = random.randint(0, 2147483647) if seed is None or int(seed) == -1 else int(seed)
    seeds = [base_seed + i for i in range(num_images)]

    try:
        results = [None] * num_images
        # แบ่งงานให้ GPU แต่ละตัวทำขนานกัน (เช่น 4 ภาพ -> GPU 0 ทำ 2 ภาพ, GPU 1 ทำ 2 ภาพ)
        def worker(gpu_idx):
            for idx in range(gpu_idx, num_images, len(pipes)):
                results[idx] = _render_single(
                    gpu_idx, prompt, negative_prompt, width, height, steps, cfg_scale, seeds[idx]
                )

        with ThreadPoolExecutor(max_workers=len(pipes)) as executor:
            futures = [executor.submit(worker, g) for g in range(len(pipes))]
            for f in futures:
                f.result()

        gallery_items = [(img, f'ภาพที่ {i+1} | Seed: {s}') for i, (img, s) in enumerate(results)]
        seeds_text = ', '.join(f'ภาพ {i+1}: {s}' for i, (_, s) in enumerate(results))
        return gallery_items, seeds_text

    except torch.cuda.OutOfMemoryError as exc:
        for i in range(len(pipes)):
            with torch.cuda.device(i):
                torch.cuda.empty_cache()
        raise gr.Error('VRAM ไม่พอ - ลดขนาดภาพหรือ Steps แล้วลองอีกครั้ง') from exc

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🌸 AI วาดภาพอนิเมะ - Animagine XL 4.0 (Dual GPU)')
    with gr.Row():
        with gr.Column(scale=1):
            prompt_input = gr.Textbox(
                label='Prompt', lines=3,
                value='1girl, silver hair, blue eyes, stylish casual clothes, sunset city background, masterpiece, high score, great score, absurdres',
            )
            neg_prompt_input = gr.Textbox(
                label='Negative Prompt', lines=3,
                value='lowres, bad anatomy, bad hands, text, error, missing fingers, extra digits, fewer digits, cropped, worst quality, low quality, low score, bad score, average score',
            )
            with gr.Row():
                aspect_dropdown = gr.Dropdown(
                    choices=['จัตุรัส (1:1)', 'แนวตั้งมาตรฐาน (3:4)', 'แนวตั้ง 9:16', 'แนวนอน 16:9'],
                    value='แนวตั้งมาตรฐาน (3:4)', label='สัดส่วนภาพ',
                )
                num_images_slider = gr.Slider(1, 4, value=2, step=1, label='จำนวนภาพ (สูงสุด 4)')
            with gr.Row():
                steps_slider = gr.Slider(15, 40, value=28, step=1, label='Steps')
                cfg_slider = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label='CFG Scale')
            seed_input = gr.Number(value=-1, label='Seed เริ่มต้น (-1 = สุ่ม)', precision=0)
            generate_btn = gr.Button('วาดภาพ', variant='primary')
        with gr.Column(scale=1):
            output_gallery = gr.Gallery(label='ผลลัพธ์', columns=2, height='auto', object_fit='contain')
            seed_output = gr.Textbox(label='Seed ของแต่ละภาพ', interactive=False)

    generate_btn.click(
        fn=generate_anime_v4,
        inputs=[prompt_input, neg_prompt_input, aspect_dropdown, num_images_slider, steps_slider, cfg_slider, seed_input],
        outputs=[output_gallery, seed_output],
    )

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)